In [3]:

from pathlib import Path
import shutil

dataset_dir = Path(
    "/kaggle/input/datasets/vesaxhelili/jwst-programmable-cubes"
)

working_dir = Path("/kaggle/working/jwst-project")

assert dataset_dir.exists(), "Dataset directory not found"

shutil.copytree(
    dataset_dir,
    working_dir,
    dirs_exist_ok=True
)

print("--- JWST Project Setup ---")
print("JWST source:", (working_dir / "src/jwst/jwst_fitness.py").exists())
print("ESA simulator:", (
    working_dir /
    "esa-reference/Challenge 3 Programmable Cubes/programmable_cubes_UDP.py"
).exists())
print("Best chromosome:", (
    working_dir / "results/jwst/best_jwst_chromosome.npy"
).exists())

print("Working directory:", working_dir)


--- JWST Project Setup ---
JWST source: True
ESA simulator: True
Best chromosome: True
Working directory: /kaggle/working/jwst-project


In [4]:

import sys
import numpy as np

sys.path.insert(0, str(working_dir / "src" / "jwst"))

from jwst_fitness import evaluate_jwst

chromosome = np.array([0, 0, -1], dtype=int)

fitness = evaluate_jwst(chromosome)

print("--- Kaggle JWST Fitness Test ---")
print("Fitness:", fitness)

assert np.isfinite(fitness)
assert np.isclose(
    fitness,
    4.762962962994166e-06,
    rtol=0,
    atol=1e-10
)

print("Kaggle JWST Fitness: PASSED")


--- Kaggle JWST Fitness Test ---
Fitness: 4.762962962994166e-06
Kaggle JWST Fitness: PASSED


In [5]:

saved = np.load(
    working_dir / "results/jwst/best_jwst_chromosome.npy"
)

saved_fitness = evaluate_jwst(saved)

print("--- Existing JWST Solution ---")
print("Commands:", (len(saved) - 1) // 2)
print("ESA fitness:", saved_fitness)

assert np.isclose(
    saved_fitness,
    -0.038571111111111125,
    rtol=0,
    atol=1e-10
)

print("Saved solution verification: PASSED")


--- Existing JWST Solution ---
Commands: 300
ESA fitness: -0.038571111111111125
Saved solution verification: PASSED


In [6]:

import json
import time
from pathlib import Path

import numpy as np
from jwst_hybrid_search import hybrid_search
from jwst_fitness import evaluate_jwst

output_dir = Path("/kaggle/working/jwst-results")
output_dir.mkdir(parents=True, exist_ok=True)

# Preserve the verified solution as our baseline.
baseline = np.load(
    working_dir / "results/jwst/best_jwst_chromosome.npy"
)

best_chromosome = baseline.copy()
best_fitness = float(evaluate_jwst(best_chromosome))

best_path = output_dir / "best_jwst_chromosome.npy"
np.save(best_path, best_chromosome)

print("Starting best fitness:", best_fitness)

seeds = [10, 20, 30]
evaluations = 300
commands = 300
results = []

for seed in seeds:
    print(f"\n--- Seed {seed} ---", flush=True)

    np.random.seed(seed)
    start = time.perf_counter()

    candidate, fitness = hybrid_search(
        num_iterations=evaluations - 1,
        num_commands=commands,
        num_mutations=10,
    )

    elapsed = time.perf_counter() - start

    results.append({
        "seed": seed,
        "fitness": float(fitness),
        "evaluations": evaluations,
        "commands": commands,
        "time_seconds": elapsed,
    })

    if fitness < best_fitness:
        best_fitness = float(fitness)
        best_chromosome = candidate.copy()

        # Immediately save the improved solution.
        np.save(best_path, best_chromosome)

        print(
            "NEW BEST FITNESS:",
            best_fitness,
            flush=True,
        )

    with open(
        output_dir / "experiment_results.json",
        "w"
    ) as file:
        json.dump(
            {
                "best_fitness": best_fitness,
                "completed_runs": results,
            },
            file,
            indent=4,
        )

    print(
        f"Completed seed {seed} in {elapsed:.2f}s",
        flush=True,
    )

# Verify the saved chromosome.
saved = np.load(best_path)
verified = evaluate_jwst(saved)

assert np.isclose(
    verified,
    best_fitness,
    rtol=0,
    atol=1e-10,
)

print("\n--- Kaggle Optimization Results ---")
print("Starting fitness:", evaluate_jwst(baseline))
print("Final best fitness:", verified)
print("Commands:", (len(saved) - 1) // 2)
print("Completed seeds:", len(results))
print("Results directory:", output_dir)
print("Verification: PASSED")


Starting best fitness: -0.038571111111111125

--- Seed 10 ---
Initial fitness: -0.0005711111111111345
Initial generation attempts: 1332
Iteration 1: Improved to -0.00257111
Iteration 5: Improved to -0.00657111
Iteration 10: Improved to -0.00857111
Iteration 16: Improved to -0.01257111
Iteration 44: Improved to -0.01457111
Iteration 56: Improved to -0.01657111
Iteration 58: Improved to -0.01857111
Iteration 71: Improved to -0.02057111
Iteration 87: Improved to -0.02257111
Iteration 89: Improved to -0.02457111
Iteration 96: Improved to -0.02657111
Iteration 102: Improved to -0.02857111
Iteration 149: Improved to -0.03257111
Iteration 165: Improved to -0.03457111
Iteration 175: Improved to -0.03657111
Iteration 255: Improved to -0.03857111
Iteration 297: Improved to -0.04057111
NEW BEST FITNESS: -0.040571111111111155
Completed seed 10 in 20.63s

--- Seed 20 ---
Initial fitness: -0.0005711111111111345
Initial generation attempts: 1241
Iteration 4: Improved to -0.00457111
Iteration 18: Impr

In [12]:

import json
import time
import numpy as np
from pathlib import Path

from jwst_hybrid_search import hybrid_search
from jwst_fitness import evaluate_jwst

output_dir = Path("/kaggle/working/jwst-results")
best_path = output_dir / "best_jwst_chromosome.npy"
history_path = output_dir / "large_experiment_results.json"

assert best_path.exists(), "Best chromosome checkpoint missing"

best_chromosome = np.load(best_path)
best_fitness = float(evaluate_jwst(best_chromosome))

print("Protected starting fitness:", best_fitness)

seeds = [40, 50, 60]
evaluations = 1000
commands = 300

results = []

for seed in seeds:
    print(f"\n--- Seed {seed} ---", flush=True)

    np.random.seed(seed)
    start = time.perf_counter()

    candidate, fitness = hybrid_search(
        num_iterations=evaluations - 1,
        num_commands=commands,
        num_mutations=10,
    )

    elapsed = time.perf_counter() - start

    results.append({
        "seed": seed,
        "fitness": float(fitness),
        "evaluations": evaluations,
        "commands": commands,
        "time_seconds": elapsed,
    })

    if fitness < best_fitness:
        best_fitness = float(fitness)
        best_chromosome = candidate.copy()
        np.save(best_path, best_chromosome)
        print("NEW BEST FITNESS:", best_fitness, flush=True)

    with open(history_path, "w") as file:
        json.dump(
            {
                "best_fitness": best_fitness,
                "completed_runs": results,
            },
            file,
            indent=4,
        )

    print(
        f"Seed {seed} completed in {elapsed:.2f}s",
        flush=True,
    )

saved = np.load(best_path)
verified = evaluate_jwst(saved)

assert np.isclose(
    verified,
    best_fitness,
    rtol=0,
    atol=1e-10,
)

print("\n--- Larger JWST Experiment ---")
print("Final best fitness:", verified)
print("Completed seeds:", len(results))
print("Verification: PASSED")


Protected starting fitness: -0.046571111111111146

--- Seed 40 ---
Initial fitness: -0.00657111111111105
Initial generation attempts: 1402
Iteration 5: Improved to -0.00857111
Iteration 13: Improved to -0.01057111
Iteration 20: Improved to -0.01257111
Iteration 21: Improved to -0.01657111
Iteration 34: Improved to -0.01857111
Iteration 69: Improved to -0.02057111
Iteration 93: Improved to -0.02457111
Iteration 162: Improved to -0.02657111
Iteration 167: Improved to -0.02857111
Iteration 246: Improved to -0.03057111
Iteration 336: Improved to -0.03257111
Iteration 356: Improved to -0.03457111
Iteration 459: Improved to -0.03657111
Iteration 679: Improved to -0.04057111
Iteration 716: Improved to -0.04257111
Iteration 759: Improved to -0.04457111
Iteration 822: Improved to -0.04657111
Iteration 958: Improved to -0.04857111
NEW BEST FITNESS: -0.04857111111111109
Seed 40 completed in 65.03s

--- Seed 50 ---
Initial fitness: 0.0014288888888888901
Initial generation attempts: 1374
Iteration 

In [14]:

from pathlib import Path
import shutil
import zipfile

working_dir = Path("/kaggle/working/jwst-project")
source_dir = working_dir / "src" / "jwst"

filenames = [
    "jwst_hybrid_experiments.py",
    "jwst_refinement_search.py",
]

assert source_dir.is_dir(), "JWST project folder missing"

for filename in filenames:
    matches = list(Path("/kaggle/input").rglob(filename))
    assert matches, f"Cannot find {filename}"

    shutil.copy2(matches[0], source_dir / filename)
    print(filename, "READY")

print("JWST source update: PASSED")


jwst_hybrid_experiments.py READY
jwst_refinement_search.py READY
JWST source update: PASSED


In [16]:

import numpy as np
from jwst_fitness import evaluate_jwst

baseline_path = Path(
    "/kaggle/working/jwst-results/best_jwst_chromosome.npy"
)

assert baseline_path.exists(), "Previous checkpoint missing"

baseline = np.load(baseline_path)
fitness = float(evaluate_jwst(baseline))

print("Starting JWST fitness:", fitness)

assert np.isclose(
    fitness,
    -0.056571111111111114,
    rtol=0,
    atol=1e-10,
)

print("Baseline verification: PASSED")


Starting JWST fitness: -0.056571111111111114
Baseline verification: PASSED


In [17]:

import sys

sys.path.insert(0, str(source_dir))

from jwst_refinement_search import run_refinement

results = run_refinement(
    baseline_path=baseline_path,
    output_dir="/kaggle/working/jwst-results/refinement",
    mutation_sizes=(1, 3, 10),
    evaluations=300,
)

print("\n--- Mutation Strategy Comparison ---")

for result in results:
    print(
        f"Mutation size {result['mutation_size']}: "
        f"fitness={result['final_fitness']:.9f}, "
        f"improvements={result['improvements']}"
    )


Starting fitness: -0.056571111111111114

--- Mutation size 1 ---

--- Mutation size 3 ---
Evaluation 152: Improved to -0.058571111
NEW GLOBAL BEST: -0.05857111111111106

--- Mutation size 10 ---
Evaluation 10: Improved to -0.058571111
Evaluation 174: Improved to -0.060571111
NEW GLOBAL BEST: -0.06057111111111108

--- JWST Refinement Summary ---
Original fitness: -0.056571111111111114
Best refinement fitness: -0.06057111111111108
Total evaluations: 900
Verification: PASSED

--- Mutation Strategy Comparison ---
Mutation size 1: fitness=-0.056571111, improvements=0
Mutation size 3: fitness=-0.058571111, improvements=1
Mutation size 10: fitness=-0.060571111, improvements=2


In [18]:

from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
import numpy as np

refinement_dir = Path(
    "/kaggle/working/jwst-results/refinement"
)

best_path = refinement_dir / "best_refined_jwst.npy"
results_path = refinement_dir / "refinement_results.json"
backup_path = Path("/kaggle/working/jwst-checkpoint-v3.zip")

assert best_path.exists()
assert results_path.exists()

best = np.load(best_path)
verified = float(evaluate_jwst(best))

assert np.isclose(
    verified,
    -0.06057111111111108,
    rtol=0,
    atol=1e-10,
)

with ZipFile(backup_path, "w", ZIP_DEFLATED) as archive:
    archive.write(best_path, "best_refined_jwst.npy")
    archive.write(results_path, "refinement_results.json")

print("--- JWST Checkpoint V3 ---")
print("Best fitness:", verified)
print("Commands:", (len(best) - 1) // 2)
print("Backup:", backup_path)
print("CHECKPOINT V3: PASSED")


--- JWST Checkpoint V3 ---
Best fitness: -0.06057111111111108
Commands: 300
Backup: /kaggle/working/jwst-checkpoint-v3.zip
CHECKPOINT V3: PASSED


In [13]:

from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
import numpy as np

output_dir = Path("/kaggle/working/jwst-results")
backup_path = Path("/kaggle/working/jwst-checkpoint-v2.zip")

files = [
    output_dir / "best_jwst_chromosome.npy",
    output_dir / "experiment_results.json",
    output_dir / "large_experiment_results.json",
]

for file in files:
    assert file.exists(), f"Missing: {file}"

best = np.load(files[0])
fitness = float(evaluate_jwst(best))

assert np.isclose(
    fitness,
    -0.056571111111111114,
    rtol=0,
    atol=1e-10
)

with ZipFile(backup_path, "w", ZIP_DEFLATED) as archive:
    for file in files:
        archive.write(file, arcname=file.name)

print("--- JWST Checkpoint V2 ---")
print("Best fitness:", fitness)
print("Commands:", (len(best) - 1) // 2)
print("Backup:", backup_path)
print("Size:", backup_path.stat().st_size, "bytes")
print("CHECKPOINT V2: PASSED")


--- JWST Checkpoint V2 ---
Best fitness: -0.056571111111111114
Commands: 300
Backup: /kaggle/working/jwst-checkpoint-v2.zip
Size: 1774 bytes
CHECKPOINT V2: PASSED


In [10]:

from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED

output_dir = Path("/kaggle/working/jwst-results")
backup_path = Path("/kaggle/working/jwst-checkpoint.zip")

required_files = [
    output_dir / "best_jwst_chromosome.npy",
    output_dir / "experiment_results.json",
]

for file in required_files:
    assert file.exists(), f"Missing: {file}"

with ZipFile(backup_path, "w", ZIP_DEFLATED) as archive:
    for file in required_files:
        archive.write(file, arcname=file.name)

print("Backup created:", backup_path)
print("Size:", backup_path.stat().st_size, "bytes")
print("Files:", [file.name for file in required_files])
print("CHECKPOINT: PASSED")


Backup created: /kaggle/working/jwst-checkpoint.zip
Size: 1450 bytes
Files: ['best_jwst_chromosome.npy', 'experiment_results.json']
CHECKPOINT: PASSED


In [11]:

from IPython.display import FileLink, display

display(FileLink("/kaggle/working/jwst-checkpoint.zip"))


/kaggle/working/jwst-checkpoint.zip

In [2]:

from pathlib import Path

input_dir = Path("/kaggle/input")

print("--- Kaggle Input Files ---")

for path in sorted(input_dir.rglob("*")):
    if path.is_file():
        print(path.relative_to(input_dir))


--- Kaggle Input Files ---
datasets/vesaxhelili/jwst-programmable-cubes/esa-reference/Challenge 3 Programmable Cubes/CubeMoveset.py
datasets/vesaxhelili/jwst-programmable-cubes/esa-reference/Challenge 3 Programmable Cubes/README.md
datasets/vesaxhelili/jwst-programmable-cubes/esa-reference/Challenge 3 Programmable Cubes/data/Enterprise/Initial_Config.npy
datasets/vesaxhelili/jwst-programmable-cubes/esa-reference/Challenge 3 Programmable Cubes/data/Enterprise/Initial_Cube_Types.npy
datasets/vesaxhelili/jwst-programmable-cubes/esa-reference/Challenge 3 Programmable Cubes/data/Enterprise/Target_Config.npy
datasets/vesaxhelili/jwst-programmable-cubes/esa-reference/Challenge 3 Programmable Cubes/data/Enterprise/Target_Cube_Types.npy
datasets/vesaxhelili/jwst-programmable-cubes/esa-reference/Challenge 3 Programmable Cubes/data/ISS/Initial_Config.npy
datasets/vesaxhelili/jwst-programmable-cubes/esa-reference/Challenge 3 Programmable Cubes/data/ISS/Initial_Cube_Types.npy
datasets/vesaxhelili/j